# Testing BridGE functions to work with AoU data

This is organized by the job that is run in the main bridge.py file.

Convert plink1 file formate to plink2

```bash
# assuming you are in the base directory of this repo

# get the same version of plink2
wget https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_avx2_20260504.zip
unzip plink2_linux_avx2_20260504.zip
rm plink2_linux_avx2_20260504.zip

# set up raw directory with refdata and gwas data

# convert plink1 file into plink2 file
./plink2 --bfile data/raw/gwas_subset --make-pgen --out data/intermediate/gwas

# run some basic filtering and export the raw file to load similarily as original BridGE implementation
./plink2 -pfile data/intermediate/gwas --snps-only just-acgt --exclude-palindromic-snps --autosome --geno 0.01 --maf 0.05 --make-pgen --out data/intermediate/gwas_subset
./plink2 -pfile data/intermediate/gwas_subset --export A --out data/intermediate/gwas_subset

# other steps used in AoU analysis
# --set-all-var-ids @:#:\$r:\$a
# --hwe 0.000001 0.001 midp keep-fewhet
# --indep-pairwise 50 5 0.1
# filtered out snps not near genes 
# calculate per chrom LD
```

In [1]:
import sys
from os import path
# from datatools import plink2pkl as p2p
# from datatools import bindataa as ba
# from datatools import msigdb2pkl as msig2p
# from datatools import mapsnp2gene as snp2gene
# from datatools import snppathway as snpp
# from datatools import bpmind as bpm
# from corefuns import matrix_operations_par as ci
# from corefuns import genstats_perm as gs
# from corefuns import fdrsampleperm as fdr
# from corefuns import collectresults as cl
# import datetime


# bridge.py all possible input args
job = ''
plinkfile = 'gwas_subset'
project_dir = 'data'
genesets = 'c2.cp.v7.1' 
gene_annotation = 'glist-hg38' 
mappingDistance = 50000
minPath = 10
maxPath = 300
alpha1 = 0.05
alpha2 = 0.05
n_workers = 4
sample_perms = 1
binaryNetwork = False
snpPerms = 100
i = -1
r = 0
pval_cutoff = 0.05
fdrcut = 0.25
densitycutoff = None
ssmfile = None
model = 'combined'
snppathwayfile = 'snp_pathway_min10_max300.pkl'

## Testing Sparsity

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix, csc_matrix, coo_matrix
from sparse import GCXS as sparse_coo
import time

def benchmark_sparsity(shape, sparsity_percent, iters=100):
    """Compare sparse vs dense for different sparsity levels"""
    n_nonzero = int(shape[0] * shape[1] * (100 - sparsity_percent) / 100)
    
    # Create dense array
    dense = np.zeros(shape)
    indices = np.random.choice(shape[0] * shape[1], n_nonzero, replace=False)
    dense.flat[indices] = np.random.rand(n_nonzero)

    # Benchmark matrix-vector product
    csr_arr = csr_matrix(dense)
    t0 = time.perf_counter()
    for _ in range(iters):
        _ = dense @ dense.T
    dense_time = time.perf_counter() - t0
    dense_mem = dense.data.nbytes
    print(f"dense={dense_mem/1e6:.1f}MB")

    csc_arr = csc_matrix(dense)
    t0 = time.perf_counter()
    for _ in range(iters):
        _ = csr_arr @ csr_arr.T
    csr_time = time.perf_counter() - t0
    speedup_csr = dense_time / csr_time
    csr_mem = csr_arr.data.nbytes
    print(f"Sparsity {sparsity_percent}%: speedup={speedup_csr:.2f}x, csr={csr_mem/1e6:.1f}MB")
    
    coo_arr = coo_matrix(dense)
    t0 = time.perf_counter()
    for _ in range(iters):
        _ = csc_arr @ csc_arr.T
    csc_time = time.perf_counter() - t0
    speedup_csc = dense_time / csc_time
    csc_mem = csc_arr.data.nbytes
    print(f"Sparsity {sparsity_percent}%: speedup={speedup_csc:.2f}x, csc={csc_mem/1e6:.1f}MB")
    
    t0 = time.perf_counter()
    for _ in range(iters):
        _ = coo_arr @ coo_arr.T
    coo_time = time.perf_counter() - t0
    speedup_coo = dense_time / coo_time
    coo_mem = coo_arr.data.nbytes
    print(f"Sparsity {sparsity_percent}%: speedup={speedup_coo:.2f}x, coo={coo_mem/1e6:.1f}MB")
    
    # sparse_coo_arr = sparse_coo(dense)
    # t0 = time.perf_counter()
    # for _ in range(iters):
    #     _ = sparse_coo_arr @ sparse_coo_arr.T
    # sparse_coo_time = time.perf_counter() - t0
    # speedup_sparse_coo = dense_time / sparse_coo_time
    # sparse_coo_mem = sparse_coo_arr.data.nbytes
    # print(f"Sparsity {sparsity_percent}%: speedup={speedup_sparse_coo:.2f}x, sparse_coo={sparse_coo_mem/1e6:.1f}MB")
    
    print()

# Test at different sparsity levels
shape = (10000, 100)
for sparsity in [90, 95, 99]:  # 50, 70
    benchmark_sparsity(shape, sparsity, iters=10)


dense=8.0MB
Sparsity 90%: speedup=0.35x, csr=0.8MB
Sparsity 90%: speedup=0.35x, csc=0.8MB
Sparsity 90%: speedup=0.35x, coo=0.8MB

dense=8.0MB
Sparsity 95%: speedup=1.56x, csr=0.4MB
Sparsity 95%: speedup=1.57x, csc=0.4MB
Sparsity 95%: speedup=1.58x, coo=0.4MB

dense=8.0MB
Sparsity 99%: speedup=59.93x, csr=0.1MB
Sparsity 99%: speedup=62.01x, csc=0.1MB
Sparsity 99%: speedup=61.67x, coo=0.1MB



## DataProcess

In [2]:
job = 'DataProcess'



if job == 'DataProcess':
    print('data processing...')
    sys.stdout.flush()

    # convert plinkfile to pickle
    if plinkfile == '':
        sys.exit('plinkFile not provided')
    rawfile = f"{project_dir}/intermediate/{plinkfile}.raw"
    pvarfile = f"{project_dir}/intermediate/{plinkfile}.pvar"
    psamfile = f"{project_dir}/intermediate/{plinkfile}.psam"
    if not path.exists(rawfile) or not path.exists(pvarfile) or not path.exists(psamfile):
        sys.exit(f'plinkFiles do not exist: {rawfile}, {pvarfile}, {psamfile}')
    finalfile = f"{project_dir}/intermediate/{plinkfile}.pkl"
    # p2p.plink2pkl(pgenfile, pvarfile, psamfile, finalfile)

    # converting snp data assuming different disease models
    # ba.bindataa(project_dir, finalfile, 'r')
    # ba.bindataa(project_dir, finalfile, 'd')

    # prepare gene set information
    symbolsfile = f"{project_dir}/raw/{genesets}.symbols.gmt"
    entrezfile = f"{project_dir}/raw/{genesets}.entrez.gmt"
    if not path.exists(symbolsfile) or not path.exists(entrezfile):
        sys.exit(f'genesets do not exist: {symbolsfile}, {entrezfile}')
    # msig2p.msigdb2pkl(symbolsfile, entrezfile)

    # build relationship between snps and genes
    gene_annotation_file = f"{project_dir}/raw/{gene_annotation}"
    if not path.exists(gene_annotation_file):
        sys.exit('gene annotation file not found')
    sgmfile = f"{project_dir}/intermediate/snpgenemapping_{int(mappingDistance/1000)}kb.pkl"
    # snp2gene.mapsnp2gene(pvarfile, gene_annotation_file, mappingDistance, 'matrix', sgmfile) # matrix mode - #change

    # extract snp-pathway information
    geneset_pkl = f"{project_dir}/intermediate/{genesets}.pkl"
    # outfile = snpp.snppathway(finalfile, sgmfile, geneset_pkl, minPath, maxPath)
    # bpm.bpmind(outfile)

data processing...


### datatools/plink2pkl.py - done

In [ ]:
import pickle

import pandas as pd

from datatools import imputesnp as isnp
from classes import SNPdataclass as snpc


def assess_sparseness(df):
    """Sparsity as percentage of missing/null values"""
    
    sparsity = df.isnull().sum().sum() / (len(df) * len(df.columns))
    print(f"Sparsity (missing/nulls): {sparsity:.2%}")

    # Count zeros as sparse (common in genomics)
    sparsity = (df == 0).sum().sum() / (len(df) * len(df.columns))
    print(f"Sparsity (zeros): {sparsity:.2%}")

    # Or combine zeros and nulls
    sparsity = ((df == 0) | df.isnull()).sum().sum() / (len(df) * len(df.columns))
    print(f"Sparsity (zeros + missing): {sparsity:.2%}")
    
def plink2pkl(rawFile, pvarFile, psamFile, outputFile):
    """Convert plink .pgen file to pickle file format.
        
    This function extracts all information from the .pgen file and 
    separates the genotype information from the rest. It saves all 
    into a <outputFile>.pkl file.
    
    INPUTS:
    rawFile - plink.raw file
    pvarFile - plink.pvar file that associated with rawFile
    psamFile - plink.psam file that associated with rawFile
    outputFile - name for output pickle file
    
    OUTPUTS:
    <outputFile>.pkl
    The .pkl file uses an SNPdata class with the following fields:
    - rsid: snp names
    - data: genotype data
    - chr: chromosome id
    - loc: physical location
    - pheno: sample's phenotype
    - fid: sample's family id
    - pid: sample id
    - sex: sample sex
    """
    
    # Creating headers for columns reading files into dataframes.
    pvar_header = ['chrom', 'pos', 'var_id', 'ref', 'alt']
    var_df = pd.read_csv(pvarFile, sep=r"\s+", header=0, names=pvar_header, engine='python')

    psam_header = ['fid', 'iid', 'sex', 'pheno']
    sam_df = pd.read_csv(psamFile, sep=r"\s+", header=0, names=psam_header, engine='python')

    geno_df = pd.read_csv(rawFile, sep=r"\s+", header=0, engine='python')

    # need to flip 0 and 2 counts, since plink's --export A counts the ref alleles
    data = 2 - geno_df[geno_df.columns[6:]]
    assess_sparseness(data)

    # remove ref allele from the end of the rsIDs
    prev_cols = data.columns.tolist()
    new_cols = [col.split('_')[0] for col in prev_cols]
    data.columns = new_cols

    # Structuring data to be saved into pickle format.
    SNPdata = snpc.SNPclass(
        data, 
        var_df.var_id, var_df.chrom, var_df.pos,
        sam_df.pheno-1, sam_df.fid, sam_df.iid, sam_df.sex,
        )

    # Save data to pickle file.
    final = open(outputFile, 'wb')
    pickle.dump(SNPdata, final, protocol=pickle.HIGHEST_PROTOCOL)
    final.close()

plink2pkl(rawfile, pvarfile, psamfile, finalfile)

Sparsity (missing/nulls): 0.00%
Sparsity (zeros): 63.02%
Sparsity (zeros + missing): 63.02%


### datatools/bindata.py - done

In [ ]:
import pickle


def bindataa(project_dir, dataFile, expr):
    """Binarize 012 format SNP data based on dominant/recessive assumptions.

    INPUTS:
    project_dir: directory of all the project files
    dataFile - name of the data file. This .mat file consists
        a structure array SNPdata with the following fields:
        - rsid:snp names
        - data:genotype data
        - chr: chromosome id
        - loc: physical location
        - pheno: sample's phenotype
        - fid: sample's family id
        - pid: sample id
        - gender
    expr - flag used to designate dominant ('d'/'D') or recessive ('r'/'R')

    OUTPUTS:
    a pickle file SNPdataA(D or R).pkl
    """
    
    # Reading in pickle datafile
    pklin = open(dataFile, "rb")
    SNPdata = pickle.load(pklin)
    pklin.close()

    # Checking expression flag to proceed as dominant or recessive (D or R).
    if expr == 'r' or expr == 'R':
        # If recessive, set 1s to 0s, 2s to 1s, and set appropriate filename.
        filename = f"{project_dir}/intermediate/SNPdataAR.pkl"
        replace_dict = {1: 0, 2: 1}
        SNPdata.data = SNPdata.data.replace(replace_dict)
        
    elif expr == 'd' or expr == 'D':
        # If dominant, set 1s to 1s, 2s to 1s, and set appropriate filename.
        filename = f"{project_dir}/intermediate/SNPdataAD.pkl"
        replace_dict = {2: 1}
        SNPdata.data = SNPdata.data.replace(replace_dict)
        
    else:
        # Default case where expression provided was neither D or R
        print("Provide 'd'/'D' or 'r'/'R' to designate dominant/recessive.")
        return

    # TODO: this redundantly saves SNPdata class, however, it would be easy to simply
    # change the data when either dominant or recessive is needed.
    # Will need to make sure that there aren't other changes to the SNPdata class other than the data
    
    # Saving updated SNPdata in output pickle file.
    final = open(filename, 'wb')
    pickle.dump(SNPdata, final, protocol=pickle.HIGHEST_PROTOCOL)
    final.close()


In [37]:
bindataa(project_dir, finalfile, 'r')

In [38]:
bindataa(project_dir, finalfile, 'd')

### datatools/msigdb2pkl.py - done

In [ ]:
import pickle

import numpy as np
import pandas as pd

from classes import genesetdataclass as gsc


def msigdb2pkl(symbolsFile, entrezFile):
    """Convert MsigDB gene set file (.gmt) to pickle file (Python pkl).
        
    Args:
        symbolsFile: MsigDB gene set file using gene symbols (.symbols.gmt).
        entrezFile: MsigDB gene set file using gene entrez ids (.entrez.gmt).
        
    OUTPUTS:
        <symbolsFile>.pkl - This pickle file uses a geneset class with fields:
            geneset.entrezids - gene {symbol: entrezID} lookup dictionary
            geneset.gpmatrix - gene pathway binary dataframe
    """
    
    # load pathway files
    symbols_df = pd.read_csv(symbolsFile, header=None)
    symbols_df = symbols_df[0].str.split('\t', expand=True, n=2)
    symbols_df.columns = ['pathway_names', "url", "gene_names"]
    symbols_df['gene_names'] = symbols_df['gene_names'].str.split('\t')

    entrez_df = pd.read_csv(entrezFile, header=None)
    entrez_df = entrez_df[0].str.split('\t', expand=True, n=2)
    entrez_df.columns = ['pathway_names', "url", "entrez_ids"]
    entrez_df['entrez_ids'] = entrez_df['entrez_ids'].str.split('\t')

    # make gene by pathway binary matrix
    pathway_list = symbols_df['pathway_names'].tolist()
    gene_list = list(set([gene for sublist in symbols_df['gene_names'].tolist() for gene in sublist]))
    gpm = pd.DataFrame(np.zeros((len(gene_list), len(pathway_list))),
                            index=pd.Series(gene_list, name='genes'),
                            columns=pd.Series(pathway_list, name='pathway'),
                            dtype=bool)
    
    # fill out binary matrix
    for pathway in pathway_list:
        pathway_mask: pd.Series = symbols_df['pathway_names'] == pathway
        genes_in_pathway = symbols_df.loc[pathway_mask, 'gene_names'].tolist()[0]
        gpm.loc[genes_in_pathway, pathway] = True
    
    # Creating dictionary for easy lookup of entrezID by symbol.
    symboldict = {}
    for symbol_genes, entrez_ids in zip(symbols_df['gene_names'].tolist(), entrez_df['entrez_ids'].tolist()):
        for symbol, entrez_id in zip(symbol_genes, entrez_ids):
            symboldict[symbol] = int(entrez_id)

    # 6/25/26 MF - confirmed this gene by pathway matrix is correct and matches the original implementation
    # Converting data to pickle storage file with geneset class.
    geneset = gsc.genesetclass(symboldict, gpm)
    symbols_pkl_file = symbolsFile.replace(".symbols.gmt", ".pkl")
    symbols_pkl_file = symbols_pkl_file.replace("raw/", "intermediate/")
    final = open(symbols_pkl_file, 'wb')
    pickle.dump(geneset, final, protocol=pickle.HIGHEST_PROTOCOL)
    final.close()


msigdb2pkl(symbolsfile, entrezfile)

### datatools/mapsnp2gene.py - done

In [12]:
import pickle

import numpy as np
import pandas as pd


def mapsnp2gene(pvarFile, geneAnnotation, mappingDistance, option, outfile):
    """Creates snp to gene matrix in the DataFrame format and saves it to a pickle file.

    Args:
        pvarFile (str): path to Plink variant file in .pvar format.
        geneAnnotation (str): path to gene annotation file.
        mappingDistance (int): snp to gene mapping distance.
        option (str): saving mode for snp-gene map.
        outfile (str): file name for saving the results.
    """

    # Creating SNP dataframe from snp annotation file.
    pvar_header = ['chrom', 'pos', 'var_id', 'ref', 'alt']
    var_df = pd.read_csv(pvarFile, sep=r"\s+", header=0, names=pvar_header, engine='python')  # has header row
    var_df['chrom'] = pd.to_numeric(var_df['chrom'])

    # Creating gene dataframe from gene annotation file.
    gene_header = ['chrom', 'geneloc1', 'geneloc2', 'genes']
    gdf = pd.read_csv(geneAnnotation, sep=r"\s+", names=gene_header, engine='python')  # does not have a header
    gdf = gdf[gdf.chrom.apply(lambda x: x.isnumeric())]
    gdf['chrom'] = pd.to_numeric(gdf['chrom'])
    gdf.sort_values(by='chrom', inplace=True)

    # Expanding gene window by subtracting and adding from start and end loci.
    gdf['geneloc1'] = gdf['geneloc1'] - mappingDistance
    gdf['geneloc2'] = gdf['geneloc2'] + mappingDistance

    # Doing an outer join to get all genes and snp listed by chromosome.
    cdf = gdf.merge(var_df, how='outer', on='chrom')

    # keep only snps that are located between start and end loci adjusted by mappingDistance.
    cdf = cdf[(cdf['pos'] >= cdf['geneloc1']) & (cdf['pos'] <= cdf['geneloc2'])]
    
    # Creating list of unique rsids from filtered results.
    snplist = cdf['var_id'].drop_duplicates()

    # Option chosen to save to snplist.
    if (option == 'snplist'):

        # Saving SNPlist to pickle file.
        final = open(outfile, 'wb')
        pickle.dump(snplist, final, protocol=pickle.HIGHEST_PROTOCOL)
        final.close()

    # Option chosen to save to matrix.
    elif (option == 'matrix'):

        # Getting list of unique and genes from filtered results.
        genelist = cdf['genes'].drop_duplicates()

        # Creating dataframe of appropriate size, and setting labels.
        sgm = pd.DataFrame(np.zeros((len(snplist), len(genelist))),
                                index=snplist, columns=genelist, dtype=bool)

        # Setting snp-gene matrix values to true if snp is within gene window.
        for row in cdf.itertuples():
            sgm.loc[row.var_id, row.genes] = True

        # Saving snp-gene matrix to pickle file.
        final = open(outfile, 'wb')
        pickle.dump(sgm, final, protocol=pickle.HIGHEST_PROTOCOL)
        final.close()

    else:
        # Output option not recognized.
        print("Return option error, valid options are 'snplist', or 'matrix'")


mapsnp2gene(pvarfile, gene_annotation_file, mappingDistance, 'matrix', sgmfile)
# 50s

(93942710, 8)
(99283, 8)
49943


### datatools/snppathway.py - done

In [34]:
import pickle

import numpy as np
import pandas as pd
from scipy.sparse import csr_array

from classes import snpsetclass as snps


def snppathway(dataFile, sgmFile, genesets, minPath, maxPath):
    """Creates a snp-to-pathway mapping a snpset class object with following fields:
        - pathways: List of pathway names
        - spmatrix: Matrix of snp-pathway mapping (Numpy 2d array)
        - geneset: Path to geneset file in .pkl format.

    Args:
        dataFile (str): Path to the file with genotype data in the Pickle format.
        sgmFile (str): SNP to gene mapping file in the Pickle format.
        genesets (str): Gene-set file in pickle format.
        minPath (int): Minimum size for a pathway to be in the mapping.
        maxPath (int): Maximum size for a pathway to be in the mapping.

    Returns:
        str: Path to the output pickle file containing the snp-to-pathway mapping.
    """
    
    # find project directory
    p_dir = dataFile.split('/')
    s = '/'
    project_dir = s.join(p_dir[0:-1])

    # Loading pickle files into objects
    pklin = open(dataFile, "rb")
    SNPdata = pickle.load(pklin)
    pklin.close()

    pklin = open(sgmFile, "rb")
    sgm = pickle.load(pklin)  # snps are rows, genes are columns
    pklin.close()

    pklin = open(genesets, "rb")
    geneset = pickle.load(pklin)
    pklin.close()

    # find the snps in SNPdata (plink data) that are also in the snp-gene matrix
    # since the snp-gene matrix was created from the plink data, this is simply a sanity check that runs fast
    tmp_ids = np.intersect1d(SNPdata.rsid, sgm.index)  # TODO: 6/25/26 MF - I need to change this to var_id in SNPClass.py
    ind_ids = sgm.index.isin(tmp_ids)
    tmp_sgm = sgm.loc[ind_ids, :]

    # keep only pathways with total genes less than upper limit and more than lower limit
    ind = (np.sum(geneset.gpmatrix, axis=0) <= maxPath) & (np.sum(geneset.gpmatrix, axis=0) >= minPath)
    tmp_gpm = geneset.gpmatrix.loc[:, ind]

    # keep genes that are in both snp-gene and gene-pathway matrices
    keep_genes = np.intersect1d(tmp_gpm.index, tmp_sgm.columns)
    tmp2_sgm = tmp_sgm.loc[:, keep_genes]
    tmp2_gpm = tmp_gpm.loc[keep_genes, :]

    # make snp-pathway matrix with dot product of sparse arrays (near instant computation)
    sg_sparse = csr_array(tmp2_sgm.to_numpy())
    gp_sparse = csr_array(tmp2_gpm.to_numpy())
    tmp_sgp = sg_sparse.dot(gp_sparse).toarray()

    # after matrix multiplication (dot product) there will be values greater than 1
    # set data type to bool and then back to int
    tmp_sgp = tmp_sgp.astype(bool).astype(int)
    tmp_sgp_df = pd.DataFrame(tmp_sgp,
                                index=pd.Series(tmp2_sgm.index, name='var_id'),
                                columns=pd.Series(tmp2_gpm.columns, name='pathway'))

    # remove pathways with total SNPs more than upper limit and less than lower limit
    ind = (np.sum(tmp_sgp_df, axis=0) <= maxPath) & (np.sum(tmp_sgp_df, axis=0) >= minPath)
    tmp_sgp_df = tmp_sgp_df.loc[:, ind]

    # remove snps (rows) that aren't in a pathway
    ind_rows = (np.sum(tmp_sgp_df, axis=1) == 0)
    remove_rows = tmp_sgp_df.index[ind_rows]
    tmp_sgp_df = tmp_sgp_df.drop(remove_rows, axis=0)

    # remove pathways (columns) that aren't in any snps
    ind_cols = (np.sum(tmp_sgp_df, axis=0) == 0)
    remove_cols = tmp_sgp_df.columns[ind_cols]
    tmp_sgp_df = tmp_sgp_df.drop(remove_cols, axis=1)

    # check again the SNP limit (mostly just for lower bound, but we'll keep in upper bound too)
    ind = (np.sum(tmp_sgp_df, axis=0) <= maxPath) & (np.sum(tmp_sgp_df, axis=0) >= minPath)
    tmp_sgp_df = tmp_sgp_df.loc[:, ind]

    # Preparing data and filename for pickle storage.
    pathways = tmp_sgp_df.sum(axis=0)
    snpset = snps.snpsetclass(pathways, tmp_sgp_df, genesets)
    outfilename = f"{project_dir}/snp_pathway_min{minPath}_max{maxPath}.pkl"

    # Saving data to pickle file.
    final = open(outfilename, 'wb')
    pickle.dump(snpset, final, protocol=pickle.HIGHEST_PROTOCOL)
    final.close()

    # Returning the name of the output file to be used by other modules.
    return outfilename
    # 6/25/26 MF - confirmed this snp by pathway matrix is correct BUT DOES NOT MATCH THE ORIGINAL IMPLEMENTATION
    # see AoU-run_bridge.md for details.


outfile = snppathway(finalfile, sgmfile, geneset_pkl, minPath, maxPath)

### datatools/bpmind.py - done

In [106]:
import pickle
from itertools import combinations

import numpy as np
import pandas as pd

from classes import bpmindclass as bpmc


def bpmind(snpPathwayFile):
    """Exctracts SNP indices for BPM/WPM sets. Saves a BPMind.pkl file with a bpmindclass class with fields:
        bpm - DataFrame with all BPM data (pathway names, pathway inices, SNPs in pathaways(redundants removed))
        wpm - DataFrame with all WPM data (pathway names, pathway inices, SNPs in pathaways)

    Args:
        snpPathwayFile (str): SNP-pathway mapping file in pickle format (.pkl), containing a matrix: Result of the snppathway function. 
    """

    # find project directory
    p_dir = snpPathwayFile.split('/')
    s = '/'
    project_dir = s.join(p_dir[0:-1])

    # Reading in pickle datafile
    pklin = open(snpPathwayFile, "rb")
    snpset = pickle.load(pklin)
    pklin.close()

    # Retrieving pathways list from snpset
    pathways = snpset.pathways
    snpmat = snpset.spmatrix

    # Finding all possible combinations of pairs for pathway names and sizes.
    combnames = np.array(list(combinations(pathways.index, 2)))

    # Finding nonzero WPM indices
    WPMind = [ np.nonzero(snpmat[column])[0].tolist() for column in snpmat.columns ]
    wpmdata = {
        'pathway': pathways.index,
        'indsize': pathways.values,
        'ind': WPMind,
        'size': ((pathways.values * pathways.values) - pathways.values) / 2,
        }
    wpm = pd.DataFrame(wpmdata)

    # Finding BPM indices
    BPMind1, BPMind2, ind1size, ind2size = [], [], [], []
    for i in range(len(snpmat.columns)):
        p1 = snpmat.iloc[:, i].to_numpy()
        
        for j in range(i + 1, len(snpmat.columns)):
            p2 = snpmat.iloc[:, j].to_numpy()
            
            # snps in pathway 1 but not in pathway 2
            d1 = p1 - p2
            ind1 = np.where(d1 == 1)[0].tolist()
            
            # snps in pathway 2 but not in pathway 1
            d2 = p2 - p1
            ind2 = np.where(d2 == 1)[0].tolist()
            
            ind1size.append(len(ind1))
            ind2size.append(len(ind2))
            
            BPMind1.append(ind1)
            BPMind2.append(ind2)

    # Getting between pathway sizes by multiplying combination available pairs.
    if (len(pathways) > 1):
        size = np.array(ind1size) * np.array(ind2size)
        # Orienting bpm/wpm data and converting to dataframes.
        bpmdata = {
            'path1names': combnames[:, 0], 'ind1size': ind1size, 'ind1': BPMind1,
            'path2names': combnames[:, 1], 'ind2size': ind2size, 'ind2': BPMind2,
            'size': size,
            }
    else:
        bpmdata = {
            'path1names': [], 'ind1size': [],
            'path2names': [], 'ind2size': [],
            'size': [],
            }
    bpm = pd.DataFrame(bpmdata)

    # Reading bpm and wpm models into bpmind class for pickle storage.
    bpmobj = bpmc.bpmindclass(bpm, wpm)

    # Saving bpmind data to pickle file.
    final = open(project_dir+'/BPMind.pkl', 'wb')
    pickle.dump(bpmobj, final, protocol=pickle.HIGHEST_PROTOCOL)
    final.close()

bpmind(outfile)

## ComputeInteraction

In [ ]:
job = 'ComputeInteraction'



if job == 'ComputeInteraction':
    if not (model == 'RR' or model == 'RD' or model == 'DD' or model == 'combined'):
        sys.exit('wrong model')
        
    snpDataAD = f"{project_dir}/intermediate/SNPdataAD.pkl"
    if not path.exists(snpDataAD):
        sys.exit(snpDataAD + ' not found')
        
    snpDataAR = f"{project_dir}/intermediate/SNPdataAR.pkl"
    if not path.exists(snpDataAR):
        sys.exit(snpDataAR + ' not found')
        
    # if r < 0 :
    #     if model == 'combined':
    #         ci.combine(project_dir, alpha1, alpha2, n_workers, i)
    #     else:
    #         ci.run(project_dir, model, alpha1, alpha2, n_workers, i)
    # else:
    #     for i in range(r + 1):
    #         if model == 'combined':
    #             ci.combine(project_dir, alpha1, alpha2, n_workers, i)
    #         else:
    #             ci.run(project_dir, model, alpha1, alpha2, n_workers, i)

### corefuns/matrix_operations_par.py

In [ ]:
import numpy as np
import time
import math
import pandas as pd
from corefuns import HygeCache as hc
import multiprocessing as mp
from multiprocessing import sharedctypes
import sys
import pickle
from os import path
from corefuns import withinclassrand as wrand
from classes import InteractionNetwork


def print_time(o):
    print(o)
    t = time.localtime()
    current_time = time.strftime("%H:%M:%S", t)
    print(current_time)
    sys.stdout.flush()

class job_quota:
    """Class used as a parameter holder for passing parameters to the threads"""
    
    def __init__(self, population_size, case_size, control_size):
        self.population_size = population_size
        self.case_size = case_size
        self.control_size = control_size
        self.symmetric = True
        self.sx = []
        self.sy = []
        self.i1 = 0
        self.i2 = 0
        self.pheno = []
        self.case_flag = True
        self.alpha1 = 0
        self.alpha2 = 0
        self.shared_risk = None
        self.shared_protective = None

def init_worker(r, p):
    """Intialize function for parallel threads"""
    
    # defining global arrays for threads to write their results in
    global shared_risk_c
    shared_risk_c = r
    global shared_protective_c
    shared_protective_c = p


def matrix_to_array(m, i, symmetric):
    """takes a 2D matrix as an input, returns the lower triangle as an 1D array"""
    
    x = m.shape[0]
    y = m.shape[1]
    # i2 = i + x  # 6/23/26 MF - unused variable
    if symmetric:
        idx_low = np.tril_indices(x, i-1, y)
        a = m[idx_low]
    else:
        a = np.reshape(m, (x * y))
    return a

def array_to_matrix(a, n, m, i, symmetric):
    """takes a 1D array as an input(lower triangle), returns the 2D array"""
    
    if symmetric:
        idx = np.tril_indices(n + i, -1)
        t = i * (i-1) / 2
        t = int(t)
        d0 = idx[0][t:]
        d1 = idx[1][t:]
        idx = (d0, d1)
        matrix = np.zeros((m, m))
        matrix[idx] = a
        matrix = matrix[i:(i + n), :]
        #matrix = np.maximum(matrix,matrix.transpose())
    else:
        matrix = np.reshape(a, (n, m))
    return matrix

def parallel_run(job_arg):
    """function for running the computation job parallelly"""
    
    sys.stdout.flush()
    # sub-matrix sx, sub-matrix sy, pheno
    pheno = job_arg.pheno
    sx = job_arg.sx
    sy = job_arg.sy
    i1 = job_arg.i1
    i2 = job_arg.i2
    symmetric_flag = job_arg.symmetric
    pheno_res = np.ones(pheno.shape) - pheno
    s = sy.shape[1]
    shared_risk = np.ctypeslib.as_array(shared_risk_c)
    shared_protective = np.ctypeslib.as_array(shared_protective_c)
    print('in the parallel run:')
    print(f'i1 = {i1} , i2 = {i2}')
    print(f'i1 = {i1} creating matrices')
    sys.stdout.flush()	

    ## reshaping pheno
    pheno = np.reshape(pheno.values, (pheno.shape[0], 1))
    pheno_res = np.reshape(pheno_res.values, (pheno_res.shape[0], 1))

    # pairwise
    ### P11
    #### Risk
    tempx = sx[:] * pheno[:]
    x11 = np.matmul(tempx.transpose(), sy)

    #### Protective
    tempx_r = sx[:] * pheno_res[:]
    xp11 = np.matmul(tempx_r.transpose(), sy)
    
    ### Genotype size for 1-1
    g11 = np.matmul(sx.transpose(),sy)
    ### 1-snps
    Ix = np.ones(sx.shape)
    Iy = np.ones(sy.shape)

    sx_res = np.subtract(Ix, sx)
    sy_res = np.subtract(Iy, sy)


    ### P00
    #### Risk
    temp = sx_res[:] * pheno[:]
    x00 = np.matmul(temp.transpose(), sy_res)
    #### Protective
    temp_r = sx_res[:] * pheno_res[:]
    xp00 = np.matmul(temp_r.transpose(), sy_res)

    ### Genotype size for 0-0
    g00 = np.matmul(sx_res.transpose(), sy_res)



    ### P10
    #### Risk
    x10 = np.matmul(tempx.transpose(), sy_res)
    #### Protective
    xp10  = np.matmul(tempx_r.transpose(), sy_res)

    ### Genotype size for 1-0
    g10 = np.matmul(sx.transpose(), sy_res)


    ### P01
    #### Risk
    x01 = np.matmul(temp.transpose(), sy)
    #### Protective
    xp01 = np.matmul(temp_r.transpose(), sy)

    ### Genotype size for 0-1
    g01 = np.matmul(sx_res.transpose(), sy)


    g11 = matrix_to_array(g11, i1, symmetric_flag)
    g10 = matrix_to_array(g10, i1, symmetric_flag)
    g01 = matrix_to_array(g01, i1, symmetric_flag)
    g00 = matrix_to_array(g00, i1, symmetric_flag)
    x00 = matrix_to_array(x00, i1, symmetric_flag)
    x11 = matrix_to_array(x11, i1, symmetric_flag)
    x01 = matrix_to_array(x01, i1, symmetric_flag)
    x10 = matrix_to_array(x10, i1, symmetric_flag)
    xp00 = matrix_to_array(xp00, i1, symmetric_flag)
    xp11 = matrix_to_array(xp11, i1, symmetric_flag)
    xp01 = matrix_to_array(xp01, i1, symmetric_flag)
    xp10 = matrix_to_array(xp10, i1, symmetric_flag)
    

    cache = hc.HygeCache(job_arg.population_size, job_arg.case_size, job_arg.control_size)

    ## for risk associated
    p11 = cache.apply_hyge(g11, x11, True)
    p01 = cache.apply_hyge(g01, x01, True)
    p10 = cache.apply_hyge(g10, x10, True)
    p00 = cache.apply_hyge(g00, x00, True)
    eps = 0.0000000001
    p11 = p11 + eps
    p10 = p10 + eps
    p01 = p01 + eps
    p00 = p00 + eps
    del x11
    del x01
    del x10
    del x00
    q = np.stack((p01, p10, p00))
    q_min = np.amin(q, 0)

    raw_out = np.divide(p11, q_min)
    log_out = np.log10(raw_out)
    log_out = np.multiply(log_out, -1)
    # index for those not passing alpha1 filter
    id1 = p11 > job_arg.alpha1
    # index for those not passing alpha2 filter
    id10 = p10 <= job_arg.alpha2
    id01 = p01 <= job_arg.alpha2
    id00 = p00 <= job_arg.alpha2

    idx = id1 | id10 | id01 | id00

    log_out[idx] = 0
    log_out[p11 == 0] = 0
    log_out[q_min == 0] = 0
    log_out[log_out < 0] = 0
    ## convert 1D result to 2D matrix
    matrix = array_to_matrix(log_out, (i2-i1), s, i1, symmetric_flag)

    #result = job_result()
    shared_risk[i1:i2, :] = matrix
    del matrix
    del log_out

    ## for protective
    p11 = cache.apply_hyge(g11, xp11, False)
    p01 = cache.apply_hyge(g01, xp01, False)
    p10 = cache.apply_hyge(g10, xp10, False)
    p00 = cache.apply_hyge(g00, xp00, False)
    p11 = p11 + eps
    p10 = p10 + eps
    p01 = p01 + eps
    p00 = p00 + eps
    del xp11
    del xp10
    del xp01
    del xp00
    
    q = np.stack((p01, p10, p00))
    q_min = np.amin(q, 0)
    raw_out = np.divide(p11, q_min)
    log_out = np.log10(raw_out)
    log_out = np.multiply(log_out, -1)

    # index for those not passing alpha1 filter
    id1 = p11 > job_arg.alpha1

    # index for those not passing alpha2 filter
    id10 = p10 <= job_arg.alpha2
    id01 = p01 <= job_arg.alpha2
    id00 = p00 <= job_arg.alpha2

    idx = id1 | id10 | id01 | id00

    log_out[idx] = 0
    log_out[p11 == 0] = 0
    log_out[q_min == 0] = 0
    log_out[log_out < 0] = 0
    ## convert 1D result to 2D matrix
    matrix = array_to_matrix(log_out, (i2-i1), s, i1, symmetric_flag)
    shared_protective[i1:i2,:] = matrix
    del log_out
    del matrix
    #end of function

In [ ]:
# run
def run(project_dir, model, alpha1, alpha2, n_workers, R):
    print('computing interaction. R='+str(R)+' model = '+model)
    ## output name
    output_name = f"{project_dir}/intermediate/ssM_mhygessi_{model}_R{R}.pkl"
    cluster_file = f"{project_dir}/intermediate/PlinkFile.cluster2"

    # loading and reading SNP data
    pkl_d = open(f"{project_dir}/intermediate/SNPdataAD.pkl", "rb")
    pkl_r = open(f"{project_dir}/intermediate/SNPdataAR.pkl", "rb")
    snpdata_d = pickle.load(pkl_d)
    snpdata_r = pickle.load(pkl_r)
    pkl_d.close()
    pkl_r.close()
    pheno = snpdata_r.pheno
    dataR = snpdata_r.data
    dataD = snpdata_d.data

    if model == 'RR':
        datai = dataR
        dataj = dataR
    elif model == 'DD':
        datai = dataD
        dataj = dataD
    else:
        datai = dataR
        dataj = dataD
    
    population_size = pheno.shape[0]
    ## shuffle phenotypes if R != 0
    if R > 0:
        if not path.exists(cluster_file):
            np.random.seed(66754)
            for i in range(R):
                permuted_idx = np.random.permutation(population_size)
            pheno = pheno[permuted_idx]
        else:
            pheno = wrand.withinclassrand(R, cluster_file, f"{project_dir}/intermediate/SNPdataAD.pkl")

    case_size = np.count_nonzero(pheno)
    control_size = population_size - case_size
    # pheno_res = np.ones(pheno.shape) - pheno  # 6/23/26 MF - unused variable
    sx = datai
    sy = dataj
    sx = pd.DataFrame(data=sx)
    sy = pd.DataFrame(data=sy)
    pheno = pd.DataFrame(data=pheno)
    s = sx.shape[1]

    ## dividing sx for parallel computing
    idx = [0]
    if model == 'RR' or model == 'DD':
        share = s*s / n_workers
        for i in range(n_workers):
            if i == n_workers -1 :
                idx.append(s)
            else:
                idx.append(math.floor(math.sqrt(idx[i] * idx[i] + share)))
    else:
        share = math.floor(s / n_workers)
        for i in range(n_workers):
            if i == n_workers - 1:
                idx.append(s)
            else:
                idx.append(idx[i] + share)

    # assign job arguments
    job_args = []
    result_risk = np.ctypeslib.as_ctypes(np.zeros((s, s)))
    result_protective = np.ctypeslib.as_ctypes(np.zeros((s, s)))
    shared_risk = sharedctypes.RawArray(result_risk._type_, result_risk)
    shared_protective = sharedctypes.RawArray(result_protective._type_, result_protective)
    for i in range(n_workers):
        job_arg = job_quota(population_size, case_size, control_size)
        job_arg.alpha1 = alpha1
        job_arg.alpha2 = alpha2
        if model == 'RR' or model == 'DD':
            job_arg.symmetric = True
        else:
            job_arg.symmetric = False
        job_arg.i1 = idx[i]
        job_arg.i2 = idx[i+1]
        job_arg.sx = sx.values[:, job_arg.i1:job_arg.i2]
        job_arg.sy = sy.values
        job_arg.pheno = pheno
        job_args.append(job_arg)


    ## creating parallel pool
    pool = mp.Pool(processes=n_workers, initializer=init_worker, initargs=(shared_risk, shared_protective))
    results = pool.map(parallel_run, job_args)

    # retrieving the 2d result arrays
    result_risk = np.ctypeslib.as_array(shared_risk)
    result_protective = np.ctypeslib.as_array(shared_protective) 

    ## copy over diagonal for DD - RR
    if model == 'RR' or model == 'DD':
        i_upper = np.triu_indices(s, 1)
        result_risk[i_upper] = result_risk.T[i_upper]
        result_protective[i_upper] = result_protective.T[i_upper]
    else:  # choose maximum, make diagonals 0
        d = np.diag_indices(s)
        i_upper = np.triu_indices(s, 1)
        #i_down = np.tril_indices(s,-1)
        up_risk = result_risk[i_upper]
        down_risk = result_risk.T[i_upper]
        up_protective = result_protective[i_upper]
        down_protective = result_protective.T[i_upper]
        r = np.stack((up_risk,down_risk))
        p = np.stack((up_protective,down_protective))
        r = np.amax(r,0)
        p = np.amax(p,0)
        result_risk[i_upper] = r
        result_risk.T[i_upper] = r
        result_risk[d] = 0
        result_protective[i_upper] = p
        result_protective.T[i_upper] = p
        result_protective[d] = 0
    network = InteractionNetwork.InteractionNetwork(result_risk,result_protective,None,None)

    # Save data to pickle file.
    final = open(output_name, 'wb')
    pickle.dump(network, final,protocol=4)
    final.close()



In [ ]:
# combine

def combine(project_dir, alpha1, alpha2, n_workers,R):
    output_name = f"{project_dir}/intermediate/ssM_mhygessi_combined_R{R}.pkl"
    ## run for 3 models
    run(project_dir, 'RR', alpha1, alpha2, n_workers, R)
    run(project_dir, 'RD', alpha1, alpha2, n_workers, R)
    run(project_dir, 'DD', alpha1, alpha2, n_workers, R)

    ## load results for 3 models
    rr_file = f"{project_dir}/intermediate/ssM_mhygessi_RR_R{R}.pkl"
    rd_file = f"{project_dir}/intermediate/ssM_mhygessi_RD_R{R}.pkl"
    dd_file = f"{project_dir}/intermediate/ssM_mhygessi_DD_R{R}.pkl"
    pklin = open(rr_file, 'rb')
    rr_network = pickle.load(pklin)
    pklin.close()
    pklin = open(rd_file, 'rb')
    rd_network = pickle.load(pklin)
    pklin.close()
    pklin = open(dd_file, 'rb')
    dd_network = pickle.load(pklin)
    pklin.close()

    s = rr_network.risk.shape[0]

    ## compare netowrks, take the element-wise max
    risk_max_id = np.zeros((s, s))
    risk_max_id[rr_network.risk > dd_network.risk ] = 1
    risk_max_id[risk_max_id == 0] = 2
    risk_max_temp = np.maximum(rr_network.risk, dd_network.risk)
    risk_max_id[risk_max_temp < rd_network.risk] = 3
    risk_max = np.maximum(risk_max_temp, rd_network.risk)
    risk_max_id[risk_max == 0] = 0

    protective_max_id = np.zeros((s, s))
    protective_max_id[rr_network.protective > dd_network.protective] = 1
    protective_max_id[protective_max_id == 0] = 2
    protective_max_temp = np.maximum(rr_network.protective, dd_network.protective)
    protective_max_id[protective_max_temp < rd_network.protective] = 3
    protective_max = np.maximum(protective_max_temp, rd_network.protective)
    protective_max_id[protective_max == 0] = 0

    ## save in the pickle format
    network = InteractionNetwork.InteractionNetwork(risk_max, protective_max, risk_max_id, protective_max_id)
    final = open(output_name, 'wb')
    pickle.dump(network, final, protocol=4)
    final.close()

## ComputeStats

In [ ]:
job = 'ComputeStats'



if job == 'ComputeStats':
    if not (model == 'RR' or model == 'RD' or model == 'DD' or model == 'combined' or ssmfile != None):
        sys.exit('wrong model')
        
    bpmfile = f"{project_dir}/intermediate/BPMind.pkl"
    if not path.exists(bpmfile):
        sys.exit(f"{bpmfile} not found")
        
    snpDataAD = f"{project_dir}/intermediate/SNPdataAD.pkl"
    if not path.exists(snpDataAD):
        sys.exit(snpDataAD + ' not found')
        
    snpDataAR = f"{project_dir}/intermediate/SNPdataAR.pkl"
    if not path.exists(snpDataAR):
        sys.exit(snpDataAR + ' not found')
        
    if ssmfile == None:
        if r < 0:
            if model == 'combined':
                ssmfile = f"{project_dir}/intermediate/ssM_mhygessi_combined_R{str(i)}.pkl"
            else:
                ssmfile = f"{project_dir}/intermediate/ssM_mhygessi_{model}_R{str(i)}.pkl"
            # gs.genstats(ssmfile, bpmfile, binaryNetwork, snpPerms, minPath, n_workers, densitycutoff)
            
        else:
            for i in range(r+1):
                if model == 'combined':
                    ssmfile = f"{project_dir}/intermediate/ssM_mhygessi_combined_R{str(i)}.pkl"
                else:
                    ssmfile = f"{project_dir}/intermediate/ssM_mhygessi_{model}_R{str(i)}.pkl"
                # gs.genstats(ssmfile, bpmfile, binaryNetwork, snpPerms, minPath, n_workers, densitycutoff)
                
    else:
        ssmfile = f"{project_dir}/intermediate/{ssmfile}"
        # gs.genstats(ssmfile, bpmfile, binaryNetwork, snpPerms, minPath, n_workers, densitycutoff)

### corefuns/genstats_perm.py

In [ ]:
# genstats


## ComputeFDR

In [ ]:
job = 'ComputeFDR'



if job == 'ComputeFDR':
    bpmfile = f"{project_dir}/intermediate/BPMind.pkl"
    if not path.exists(bpmfile):
        sys.exit(f"{bpmfile} not found")
        
    if ssmfile == None:
        if model == 'combined':
            ssmfile = f"{project_dir}/intermediate/ssM_mhygessi_combined_R0.pkl"
        else:
            ssmfile = f"{project_dir}/intermediate/ssM_mhygessi_{model}_R0.pkl"
    else:
        ssmfile = f"{project_dir}/intermediate/{ssmfile}"
    if not path.exists(ssmfile):
        sys.exit(f"{ssmfile} not found")
        
    # fdr.fdrsampleperm(ssmfile, bpmfile, pval_cutoff, minPath, sample_perms)

### corefuns/fdrsampleperm

In [ ]:
# fdrsampleperm


## Summarize

In [ ]:
job = 'Summarize'



if job == 'Summarize':
    bpmfile = f"{project_dir}/intermediate/BPMind.pkl"
    if not path.exists(bpmfile):
        sys.exit(f"bpm file not found at: {bpmfile}")
        
    snppathwayfile = f"{project_dir}/intermediate/{snppathwayfile}"
    if not path.exists(snppathwayfile):
        sys.exit(f"snp-pathway mapping file not found at: {snppathwayfile}")
        
    snpgenemappingfile = f"{project_dir}/intermediate/snpgenemapping_{int(mappingDistance/1000)}kb.pkl"
    if not path.exists(snpgenemappingfile):
        sys.exit(f"snpgenemappingfile not found at: {snpgenemappingfile}")
    
    if ssmfile == None:
        imported = False
        if model == 'combined':
            ssmfile = f"{project_dir}/intermediate/ssM_mhygessi_combined_R0.pkl"
            resultsfile = f"{project_dir}/intermediate/results_ssM_mhygessi_combined_R0.pkl"
        else:
            ssmfile = f"{project_dir}/intermediate/ssM_mhygessi_{model}_R0.pkl"
            resultsfile = f"{project_dir}/intermediate/results_ssM_mhygessi_{model}_R0.pkl"
    else:
        resultsfile = f"{project_dir}/intermediate/results_{ssmfile}"
        ssmfile = f"{project_dir}/intermediate/{ssmfile}"
        imported = True
    if not path.exists(ssmfile):
        sys.exit(f"interaction file not found at: {ssmfile}")
    if not path.exists(resultsfile):
        sys.exit(f"results file not found at: {resultsfile}")

    # cl.collectresults(resultsfile, fdrcut, ssmfile, bpmfile, snppathwayfile, snpgenemappingfile, imported, densitycutoff)


### corefuns/collectresults

In [ ]:
# collectresults
